<a href="https://colab.research.google.com/github/Marfall/AnomalyDetection-Otus-3/blob/main/AnomalyDetection_Otus_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание №3 - Поиск Анамалий в Данных

##1. Введение

Домашнее задание посвящено обнаружению аномалий в банковских транзакциях.  
Датасет creditcard.csv содержит анонимизированные признаки (первые 28 главных компонент), а также столбцы Time и Amount. Целевая переменная Class указывает, является ли транзакция мошеннической (1) или обычной (0).  
Аномалий менее 1%, что делает задачу сложной для unsupervised методов.

Цель: обучить три модели (Isolation Forest, LOF, One‑Class SVM) и оценить их способность находить мошеннические операции.

In [ ]:
# 2. Установка дополнительных библиотек (UMAP не входит в стандартный набор)
!pip install -q umap-learn
print("Библиотеки установлены.")

##3. Импорт библиотек

Загружаются все необходимые пакеты: pandas, numpy, matplotlib, seaborn, sklearn, umap.
Настраивается стиль графиков.

In [ ]:
# 4. Импорт библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap.umap_ as umap

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9

print("✅ Библиотеки загружены.")

##5. Загрузка данных

Скрипт ищет файл creditcard.csv в текущей папке и в /content/.  
Если файл не найден, будет предложено загрузить его вручную через диалог.

In [ ]:
# 6. Загрузка данных (локальный поиск + ручной аплоад)
possible_names = ['creditcard.csv']
file_path = None

for name in possible_names:
    if os.path.exists(name):
        file_path = name
        break
    if os.path.exists('/content/' + name):
        file_path = '/content/' + name
        break

if file_path is None:
    print("❌ Файл creditcard.csv не найден.")
    print("➡️ Загрузите файл вручную.")
    from google.colab import files
    uploaded = files.upload()
    file_path = list(uploaded.keys())[0]
    print(f"✅ Файл '{file_path}' загружен.")

df = pd.read_csv(file_path)
print(f"✅ Данные загружены, форма: {df.shape}")
print("Первые 5 строк:")
display(df.head())

##7. Первичный осмотр данных (EDA)

Выводятся описательные статистики, информация о колонках, распределение классов.  
Строятся гистограммы для первых пяти признаков (V1–V5).  
Рассчитывается доля аномалий (класс 1) – это значение будет использовано как параметр contamination для моделей.

In [ ]:
# 8. EDA: статистика, распределение, гистограммы
# Удаление пропусков (на всякий случай)
if df['Class'].isnull().any():
    print("⚠️ В 'Class' есть пропуски. Удаляем.")
    df = df.dropna(subset=['Class'])
if df.isnull().any().any():
    print("⚠️ Есть пропуски в других колонках. Удаляем.")
    df = df.dropna()

print("\n=== ОПИСАТЕЛЬНАЯ СТАТИСТИКА ===")
display(df.describe())

print("\n=== ИНФОРМАЦИЯ О КОЛОНКАХ ===")
df.info()

class_counts = df['Class'].value_counts()
print("\n=== РАСПРЕДЕЛЕНИЕ КЛАССОВ ===")
print(class_counts)
contamination = class_counts[1] / len(df)
print(f"Доля аномалий: {contamination*100:.4f}%")
print(f"contamination = {contamination:.6f}")

# Гистограммы
print("\n=== ГИСТОГРАММЫ ПРИЗНАКОВ V1-V5 ===")
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, col in enumerate(df.columns[:5]):
    ax = axes[i//3, i%3]
    ax.hist(df[col], bins=50, alpha=0.7, color='blue', edgecolor='black')
    ax.set_title(col)
    ax.set_xlabel('Значение')
    ax.set_ylabel('Частота')
plt.tight_layout()
plt.show()
plt.close('all')

##9. Подготовка данных к обучению

Целевая переменная Class отделяется от признаков.  
Все признаки масштабируются с помощью StandardScaler – это необходимо для корректной работы методов, основанных на расстояниях.  
Для визуализации (t‑SNE, UMAP) берётся случайная выборка из 10 000 точек, чтобы ускорить вычисления. Фиксируется random_state для воспроизводимости.

In [ ]:
# 10. Масштабирование и выборка для визуализации
y = df['Class'].values
X = df.drop(columns=['Class']).values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("✅ Масштабирование выполнено.")

sample_size = 10000
np.random.seed(42)
indices = np.random.choice(X_scaled.shape[0], sample_size, replace=False)
X_sample = X_scaled[indices]
y_sample = y[indices]
print(f"Выборка для визуализации: {sample_size} точек (аномалий: {sum(y_sample)})")

##11. Обучение моделей и оценка качества

Используются три алгоритма:
- Isolation Forest – основан на случайных разбиениях.
- Local Outlier Factor (LOF) – основан на плотности.
- One‑Class SVM – метод опорных векторов для одного класса.

Параметр contamination устанавливается равным реальной доле аномалий (экспертная оценка).  
Для каждой модели вычисляются:
- Classification report (precision, recall, f1-score)
- Confusion matrix
- ROC‑AUC (если модель позволяет получить scores)

In [ ]:
# 12. Обучение и оценка моделей
models = {
    'Isolation Forest': IsolationForest(contamination=contamination, random_state=42),
    'Local Outlier Factor': LocalOutlierFactor(contamination=contamination, novelty=False),
    'One-Class SVM': OneClassSVM(nu=contamination, kernel='rbf', gamma='scale')
}

print("\n" + "="*60)
print("ОЦЕНКА МОДЕЛЕЙ")
print("="*60)

for name, model in models.items():
    print(f"\n--- {name} ---")

    if name == 'Local Outlier Factor':
        y_pred_raw = model.fit_predict(X_scaled)
    else:
        model.fit(X_scaled)
        y_pred_raw = model.predict(X_scaled)

    y_pred = np.where(y_pred_raw == -1, 1, 0)

    print(classification_report(y, y_pred, target_names=['Норма', 'Аномалия']))
    print("Confusion matrix:")
    print(confusion_matrix(y, y_pred))

    if name == 'Isolation Forest':
        scores = model.decision_function(X_scaled)
        roc_auc = roc_auc_score(y, -scores)
        print(f"ROC-AUC: {roc_auc:.4f}")
    elif name == 'Local Outlier Factor':
        scores = model.negative_outlier_factor_
        roc_auc = roc_auc_score(y, -scores)
        print(f"ROC-AUC: {roc_auc:.4f}")
    elif name == 'One-Class SVM':
        scores = model.decision_function(X_scaled)
        roc_auc = roc_auc_score(y, -scores)
        print(f"ROC-AUC: {roc_auc:.4f}")

##13. Визуализация в двумерном пространстве

Для наглядного представления данных используются два метода уменьшения размерности:
- t‑SNE – нелинейный, хорошо разделяет кластеры.
- UMAP – современный, быстрый аналог t‑SNE.

На графиках точки окрашены в зависимости от истинного класса (норма – синий, аномалия – красный).  
Это позволяет визуально оценить, насколько аномалии отделены от основной массы.

In [ ]:
# 14. Построение t-SNE и UMAP
print("\n" + "="*60)
print("ВИЗУАЛИЗАЦИЯ В 2D")
print("="*60)

print("t-SNE...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=500)
X_tsne = tsne.fit_transform(X_sample)
print("t-SNE готово.")

print("UMAP...")
umap_model = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
X_umap = umap_model.fit_transform(X_sample)
print("UMAP готово.")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
scatter = ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_sample, cmap='coolwarm', s=5, alpha=0.7)
ax.set_title('t-SNE проекция')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.legend(*scatter.legend_elements(), title='Class', labels=['Норма', 'Аномалия'])
ax.grid(True)

ax = axes[1]
scatter = ax.scatter(X_umap[:, 0], X_umap[:, 1], c=y_sample, cmap='coolwarm', s=5, alpha=0.7)
ax.set_title('UMAP проекция')
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.legend(*scatter.legend_elements(), title='Class', labels=['Норма', 'Аномалия'])
ax.grid(True)

plt.tight_layout()
plt.show()
plt.close('all')

##15. Интерпретация результатов

Ниже приведены ключевые метрики для каждой модели (получены при запуске кода):

| Модель               | ROC‑AUC | Precision (аном.) | Recall (аном.) | F1 (аном.) | Комментарий |
|----------------------|---------|-------------------|----------------|------------|-------------|
| Isolation Forest     | 0.9451  | 0.26              | 0.26           | 0.26       | Лучший баланс, находит 126 из 492 аномалий |
| Local Outlier Factor | 0.5127  | 0.00              | 0.00           | 0.00       | Полностью не работает |
| One‑Class SVM        | 0.9353  | 0.09              | 0.28           | 0.14       | Высокий recall, но много ложных срабатываний |

**Выводы:**
- Isolation Forest показывает наилучшее качество (ROC‑AUC 0.9451) и может быть рекомендован для практического использования.
- Local Outlier Factor неприменим для этого датасета (вероятно, из‑за высокой размерности и разреженности данных).
- One‑Class SVM даёт больше найденных аномалий, но ценой огромного числа ложных тревог – использовать не рекомендуется.
- На визуализациях (t‑SNE, UMAP) видно, что аномалии часто отделены от основной массы точек, но не всегда чётко, что объясняет сложность задачи.

**Рекомендации:**
- Для улучшения качества можно использовать ансамбль моделей или калибровку порога.
- В реальном проекте важно настраивать порог классификации в зависимости от бизнес-требований (cost/benefit анализа).

##16. Итоговые выводы

1. Доля аномалий в данных: 0.1727% (492 из 284807 транзакций).
2. Лучшая модель: Isolation Forest (ROC‑AUC = 0.9451).
3. LOF показал неработоспособность – все аномалии были пропущены.
4. One‑Class SVM нашёл 138 аномалий, но дал 1345 ложных срабатываний.
5. Визуализация подтверждает, что аномалии в основном отделены, но не всегда очевидны.
6. Рекомендация: использовать Isolation Forest с возможной настройкой порога, либо ансамбль.

Таким образом, unsupervised методы способны обнаруживать мошеннические транзакции, но требуют тщательной настройки и оценки.